In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Change these to your own local file paths
MODEL_PATH = "/root/autodl-tmp/Qwen/Qwen3___5-9B"
DATA_PATH = "/root/autodl-tmp/train_dataset_updated.json"
OUTPUT_DIR = "/root/autodl-tmp/Qwen3.5-9B-SFT-Adapter"

MAX_SEQ_LENGTH = 8192        
PER_DEVICE_BATCH_SIZE = 1    
GRADIENT_ACCUMULATION = 16   
LEARNING_RATE = 2e-4
NEFTUNE_ALPHA = 5.0          

print("Loading tokenizer and dataset")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def format_and_tokenize(example):
    system_instruction = "You are a highly efficient memory extraction assistant. {\"enable_thinking\": false}"
    instruction = example.get('instruction', '').strip()
    user_input = example.get('input', '').strip()
    assistant_output = example.get('output', '').strip()
    source_theme = example.get('_source_theme', '未分类').strip()
    
    user_content = f"【文本主题】: {source_theme}\n【任务指令】: {instruction}\n【文本内容】:\n{user_input}"
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_output}
    ]
    
    prompt_messages = messages[:-1]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    
    encoded_full = tokenizer(full_text, max_length=MAX_SEQ_LENGTH, truncation=True, padding=False)
    encoded_prompt = tokenizer(prompt_text, max_length=MAX_SEQ_LENGTH, truncation=True, padding=False)
    
    prompt_len = len(encoded_prompt["input_ids"])
    labels = encoded_full["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len
    
    encoded_full["labels"] = labels
    return encoded_full

tokenized_dataset = dataset.map(format_and_tokenize, remove_columns=dataset.column_names, num_proc=4)

os.environ["BNB_CUDA_VERSION"] = "124"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":/usr/local/cuda/lib64"

# Configure 4bit bitsandbytes quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading 4bit base model")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj", 
    "gate_proj", "up_proj", "down_proj",
    "linear_q", "linear_k", "linear_v", "linear_o"
]

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,                     
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",               
    bf16=True,                              
    max_grad_norm=0.3,
    neftune_noise_alpha=NEFTUNE_ALPHA,      
    report_to="none"                        
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting QLoRA training")
trainer.train()

In [ ]:
import shutil
from peft import PeftModel

CHECKPOINT_PATH = "/root/autodl-tmp/Qwen3.5-9B-SFT-Adapter/checkpoint-186"
FINAL_SAVE_PATH = "/root/autodl-tmp/Qwen3.5-9B-SFT-Adapter/first_adapter"
MERGED_PATH = "/root/autodl-tmp/Helper"

# Collect files from training checkpoint
os.makedirs(FINAL_SAVE_PATH, exist_ok=True)
if os.path.exists(CHECKPOINT_PATH):
    for file_name in os.listdir(CHECKPOINT_PATH):
        source_file = os.path.join(CHECKPOINT_PATH, file_name)
        if os.path.isfile(source_file) and not file_name.startswith("rng_state"):
            shutil.copy(source_file, FINAL_SAVE_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

print("Loading weights for temporary merge test")
model_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
tokenizer_merge = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

model_merge = PeftModel.from_pretrained(model_merge, FINAL_SAVE_PATH)
model_merge = model_merge.merge_and_unload()

print("Saving and clearing temporary model")
model_merge.save_pretrained(MERGED_PATH)
tokenizer_merge.save_pretrained(MERGED_PATH)

if os.path.exists(MERGED_PATH):
    shutil.rmtree(MERGED_PATH)

In [ ]:
from peft import PeftModel, prepare_model_for_kbit_training

BASE_MODEL_PATH = "/root/autodl-tmp/Qwen/Qwen3___5-9B"
PREV_ADAPTER_PATH = "/root/autodl-tmp/Qwen3.5-9B-SFT-Adapter/first_adapter" 
DATA_PATH_P2 = "/root/autodl-tmp/train_dataset.json" 
OUTPUT_DIR_P2 = "/root/autodl-tmp/Qwen_Final_Adapter" 
FINAL_MERGED_PATH = "/root/autodl-tmp/Qwen3.5-Helper" 

MAX_SEQ_LENGTH_P2 = 4096

print("Processing phase 2 dataset")
tokenizer_p2 = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
tokenizer_p2.pad_token = tokenizer_p2.eos_token
dataset_p2 = load_dataset("json", data_files=DATA_PATH_P2, split="train")

def format_and_tokenize_p2(example):
    system_instruction = "You are a highly efficient memory extraction assistant. {\"enable_thinking\": false}"
    instruction = example.get('instruction', '').strip()
    user_input = example.get('input', '').strip()
    assistant_output = example.get('output', '').strip()
    source_theme = example.get('_source_theme', '未分类').strip()
    
    user_content = f"【文本主题】: {source_theme}\n【任务指令】: {instruction}\n【文本内容】:\n{user_input}"
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_output}
    ]
    
    prompt_messages = messages[:-1]
    prompt_text = tokenizer_p2.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer_p2.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    
    encoded_full = tokenizer_p2(full_text, max_length=MAX_SEQ_LENGTH_P2, truncation=True, padding=False)
    encoded_prompt = tokenizer_p2(prompt_text, max_length=MAX_SEQ_LENGTH_P2, truncation=True, padding=False)
    
    prompt_len = len(encoded_prompt["input_ids"])
    labels = encoded_full["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len 
    encoded_full["labels"] = labels
    return encoded_full

tokenized_dataset_p2 = dataset_p2.map(format_and_tokenize_p2, remove_columns=dataset_p2.column_names, num_proc=4)

print("Loading unquantized bf16 base model")
base_model_p2 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,   
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"    
)

base_model_p2.config.use_cache = False
base_model_p2.gradient_checkpointing_enable()
base_model_p2.enable_input_require_grads() 

# Resume training from the previous adapter weights
model_p2 = PeftModel.from_pretrained(base_model_p2, PREV_ADAPTER_PATH, is_trainable=True)
model_p2.print_trainable_parameters()

training_args_p2 = TrainingArguments(
    output_dir=OUTPUT_DIR_P2,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=3e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    optim="adamw_torch",                    
    bf16=True,
    max_grad_norm=0.3,
    neftune_noise_alpha=0.6,                
    report_to="none"
)

data_collator_p2 = DataCollatorForSeq2Seq(
    tokenizer=tokenizer_p2,
    model=model_p2,
    padding=True,
    pad_to_multiple_of=8
)

trainer_p2 = Trainer(
    model=model_p2,
    args=training_args_p2,
    train_dataset=tokenized_dataset_p2,
    data_collator=data_collator_p2,
)

print("Starting Phase 2 full precision LoRA training")
trainer_p2.train()

final_save_path = os.path.join(OUTPUT_DIR_P2, "ultimate_adapter")
trainer_p2.save_model(final_save_path)
tokenizer_p2.save_pretrained(final_save_path)

print("Executing final model merge and export")
base_model_final = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,  
    device_map="auto",
    trust_remote_code=True
)
tokenizer_final = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)

model_final = PeftModel.from_pretrained(base_model_final, final_save_path)
model_final = model_final.merge_and_unload()

model_final.save_pretrained(FINAL_MERGED_PATH)
tokenizer_final.save_pretrained(FINAL_MERGED_PATH)
print("Process completed successfully")